In [2]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)

import sys
sys.path.insert(0, '../')
from util import *     # preprocess_scenario, Timer, count_parameters, etc.
from models.ae import AnomalyDetector

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]
MODEL_NAME = "AnomalyAE"
TARGET_FAR = 0.05   # 5% false alarm rate on healthy samples


device: NVIDIA A30


In [3]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def run_scenario(scenario_idx, scenario_dir, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4,
                 target_far=TARGET_FAR, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)

    # split off F0 samples for unsupervised training
    f0_tr_mask = (y_tr == 0)
    f0_va_mask = (y_va == 0)
    X_tr_f0 = X_tr[f0_tr_mask]
    X_va_f0 = X_va[f0_va_mask]

    # binary labels for evaluation (0 = F0, 1 = any fault)
    y_te_binary = (y_te.numpy() != 0).astype(int)

    # --- compute metrics setup ---
    detector = AnomalyDetector(in_dim=13, hidden_dims=(32, 16),
                               latent_dim=8, device=device, seed=seed)
    n_params, params_by_type = count_parameters(detector.model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({MODEL_NAME}) ===")
        print(f"  full train: {len(X_tr)}  F0-only: {len(X_tr_f0)}  "
              f"F0-val: {len(X_va_f0)}  test: {len(X_te)}")
        print(f"  test binary labels: "
              f"F0={np.sum(y_te_binary == 0)}  fault={np.sum(y_te_binary == 1)}")
        print(f"  AE params: {n_params:,}  breakdown: {params_by_type}")

    # --- training ---
    with Timer(device) as train_timer:
        detector.fit(X_tr_f0, X_f0_val=X_va_f0, epochs=epochs, lr=lr,
                     weight_decay=weight_decay, batch_size=50,
                     seed=seed, verbose=verbose)
        # calibrate threshold on held-out F0 samples
        thresh = detector.calibrate(X_va_f0, target_far=target_far)

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    if verbose:
        print(f"  threshold (FAR={target_far}): {thresh:.6f}")

    # --- inference timing ---
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(detector.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # --- evaluation: binary only (AE cannot do multi-class) ---
    y_pred_binary = detector.predict(X_te)
    y_scores = detector.score(X_te)

    acc_bin = accuracy_score(y_te_binary, y_pred_binary)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        y_te_binary, y_pred_binary, average="binary", zero_division=0)
    auc = roc_auc_score(y_te_binary, y_scores)
    cm_bin = confusion_matrix(y_te_binary, y_pred_binary, labels=[0, 1])

    # per-class detection rate (what fraction of each fault class gets flagged?)
    # useful to see which faults are easy/hard to detect
    per_class_detection = {}
    y_te_np = y_te.numpy()
    for c in range(8):
        mask = (y_te_np == c)
        if mask.sum() > 0:
            per_class_detection[c] = float(y_pred_binary[mask].mean())

    if verbose:
        print(f"  TEST (binary): acc={acc_bin:.4f}  P={p_bin:.4f}  "
              f"R={r_bin:.4f}  F1={f1_bin:.4f}  AUC={auc:.4f}")
        print(f"  per-class flag rate: "
              f"{' '.join(f'F{c}={v:.2f}' for c,v in per_class_detection.items())}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":            scenario_idx,
        "model":               MODEL_NAME,
        "n_train_total":       len(X_tr),
        "n_train_f0":          len(X_tr_f0),
        "accuracy_binary":     acc_bin,
        "precision_binary":    p_bin,
        "recall_binary":       r_bin,
        "f1_binary":           f1_bin,
        "auc":                 auc,
        "threshold":           thresh,
        "confusion_binary":    cm_bin,
        "per_class_detection": per_class_detection,
        "n_params":            n_params,
        "train_sec":           round(train_timer.elapsed, 2),
        "inf_ms_per_sample":   round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb":         round(peak_mem_mb, 1),
    }


In [4]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, device, epochs=70, lr=1e-2, seed=0)
    results.append(r)



=== Scenario 1 (AnomalyAE) ===
  full train: 7858  F0-only: 858  F0-val: 200  test: 1600
  test binary labels: F0=200  fault=1400
  AE params: 2,229  breakdown: {'encoder': 1112, 'decoder': 1117}
  ep   1 | train MSE 0.204029 | val MSE 0.473596
  ep  10 | train MSE 0.004626 | val MSE 0.331695
  ep  20 | train MSE 0.002885 | val MSE 0.246211
  ep  30 | train MSE 0.000867 | val MSE 0.236424
  ep  40 | train MSE 0.000713 | val MSE 0.235967
  ep  50 | train MSE 0.000572 | val MSE 0.237653
  ep  60 | train MSE 0.000562 | val MSE 0.239243
  ep  70 | train MSE 0.000548 | val MSE 0.240974
  threshold (FAR=0.05): 0.003327
  TEST (binary): acc=0.6863  P=0.9880  R=0.6493  F1=0.7836  AUC=0.9459
  per-class flag rate: F0=0.06 F1=0.52 F2=0.98 F3=0.14 F4=1.00 F5=0.59 F6=0.63 F7=0.69
  COMPUTE: train=6.0s  inf=0.000ms/sample  peak_mem=17.4MB

=== Scenario 2 (AnomalyAE) ===
  full train: 3939  F0-only: 439  F0-val: 200  test: 1600
  test binary labels: F0=200  fault=1400
  AE params: 2,229  breakdown:

In [5]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train_total", "n_train_f0",
                       "accuracy_binary", "precision_binary", "recall_binary",
                       "f1_binary", "auc", "n_params", "train_sec",
                       "inf_ms_per_sample", "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy_binary", "precision_binary", "recall_binary",
              "f1_binary", "auc"]
summary[cols_round] = summary[cols_round].round(4)
print(f"\n=== {MODEL_NAME} summary across scenarios ===")
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)



=== AnomalyAE summary across scenarios ===
 scenario     model  n_train_total  n_train_f0  accuracy_binary  precision_binary  recall_binary  f1_binary    auc  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 AnomalyAE           7858         858           0.6862            0.9880         0.6493     0.7836 0.9459      2229       5.96             0.0003         17.4
        2 AnomalyAE           3939         439           0.7269            0.9829         0.7000     0.8177 0.9133      2229       1.78             0.0003         17.4
        3 AnomalyAE            786          86           0.4856            0.9816         0.4200     0.5883 0.8286      2229       0.40             0.0003         17.4
        4 AnomalyAE            391          41           0.4681            0.9757         0.4021     0.5695 0.7595      2229       0.21             0.0003         17.4
        5 AnomalyAE            238          28           0.4106            0.9790         0.3336     0.4976 0.7404  

In [6]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_f0={r['n_train_f0']}, acc_bin={r['accuracy_binary']:.3f}, "
          f"AUC={r['auc']:.3f})")
    print("Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):")
    print(r["confusion_binary"])
    print("Per-class detection rate (fraction flagged as anomaly):")
    for c, v in r["per_class_detection"].items():
        marker = "  <- NORMAL (want low)" if c == 0 else ""
        print(f"  F{c}: {v:.3f}{marker}")


Scenario 1  (AnomalyAE, n_f0=858, acc_bin=0.686, AUC=0.946)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[189  11]
 [491 909]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.055  <- NORMAL (want low)
  F1: 0.515
  F2: 0.980
  F3: 0.135
  F4: 1.000
  F5: 0.590
  F6: 0.630
  F7: 0.695

Scenario 2  (AnomalyAE, n_f0=439, acc_bin=0.727, AUC=0.913)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[183  17]
 [420 980]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.085  <- NORMAL (want low)
  F1: 0.475
  F2: 1.000
  F3: 0.215
  F4: 1.000
  F5: 0.705
  F6: 0.675
  F7: 0.830

Scenario 3  (AnomalyAE, n_f0=86, acc_bin=0.486, AUC=0.829)
Binary CM (rows=true [F0, fault], cols=pred [F0, fault]):
[[189  11]
 [812 588]]
Per-class detection rate (fraction flagged as anomaly):
  F0: 0.055  <- NORMAL (want low)
  F1: 0.465
  F2: 0.580
  F3: 0.095
  F4: 1.000
  F5: 0.615
  F6: 0.030
  F7: 0.155

Scenario 4  (AnomalyAE, n_f0=41, acc_bin=0.468, AUC=